# 03. 고성능 모델

- 담당: **③ 안주형 (고성능 모델)** (이 파일은 담당자만 수정)
- 기록 파일: `experiments/log_A_model.csv`

### 할 일
- [ ] XGBoost 개선, LightGBM·CatBoost 비교 (같은 fold, 같은 피처)
- [ ] 클래스 불균형 대응 (`balanced_weights` 등)
- [ ] 튜닝, 앙상블 후보 (cv['oof_proba'] 확률 평균)
- [ ] seed·하이퍼파라미터를 코드에 숫자로 남기기

> **규칙:** 전처리는 Pipeline 안에 / test는 보지 않기 / 점수는 `run_cv()`로 / 결과는 `log_result()`로 기록 / 자세한 규칙은 `docs/coding_rules.md`

## 0. 환경 설정 (수정하지 말고 그대로 실행)
Colab은 구글 드라이브 `내 드라이브`에 레포 폴더(`ah07-cancer-subtype`, 안에 `data/` 포함)를 두면 돼요.

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/drive/MyDrive/ah07-cancer-subtype"  # 드라이브에 둔 레포 폴더
else:
    REPO_DIR = os.path.abspath(os.path.join("..", ".."))  # notebooks/이니셜_역할/ 폴더 기준

# 경로 확인: 여기서 멈추면 폴더 위치나 이름을 확인해 주세요
assert os.path.isdir(os.path.join(REPO_DIR, "src")), (
    f"src 폴더를 찾을 수 없어요: {REPO_DIR}\n"
    "→ 드라이브 '내 드라이브' 바로 아래에 'ah07-cancer-subtype' 폴더가 있는지, "
    "ZIP을 풀 때 폴더가 두 번 겹치지 않았는지(ah07-cancer-subtype/ah07-cancer-subtype-develop) 확인")
assert os.path.isfile(os.path.join(REPO_DIR, "data", "train.csv")), (
    "data/train.csv 가 없어요 → 데이콘 csv 3개를 data/ 폴더에 넣어 주세요")
sys.path.insert(0, REPO_DIR)

from src.config import SEED, DATA_DIR, set_seed
set_seed(SEED)
print("REPO_DIR:", REPO_DIR, "| DATA_DIR:", DATA_DIR)

In [ ]:
import platform
import numpy as np, pandas as pd, sklearn
from sklearn.pipeline import Pipeline

from src.data_loader import load_data, split_xy, encode_target
from src.preprocessing import basic_transformer, binarize_mutations
from src.features import mutation_count
from src.validation import get_folds, run_cv, balanced_weights
from src.evaluation import macro_f1, per_class_f1, log_result
from src.prediction import make_submission

print("OS:", platform.platform(), "| python", platform.python_version(),
      "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

## 1. 데이터

In [ ]:
train, test = load_data(DATA_DIR)   # test는 형식 확인과 최종 추론에만
X, y_raw = split_xy(train)
y, le = encode_target(y_raw)
folds = get_folds(y)                 # 팀 공통 fold
print("train", train.shape, "| test", test.shape, "| classes", len(le.classes_))

## 실험 예시 (복사해서 바꿔 쓰세요)
1. Pipeline에 피처 변환과 모델을 넣고
2. `run_cv()`로 점수를 내고
3. `log_result()`로 기록해요.

In [ ]:
# from xgboost import XGBClassifier
# PARAMS = dict(n_estimators=200, learning_rate=0.1, max_depth=6, tree_method="hist", random_state=SEED, n_jobs=-1)
# pipe = Pipeline([("features", basic_transformer()), ("model", XGBClassifier(**PARAMS))])
# cv = run_cv(pipe, X, y, folds=folds)
# per_class_f1(y, cv["oof"], le.classes_).head(10)
# log_result(owner="A_model", notebook="03_models.ipynb", experiment="실험 이름",
#            features="basic(WT=0,변이=1)", model="XGBoost", params=PARAMS, cv=cv, notes="")